# Импорты и загрузка данных

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("data_01.csv")
print("Исходный размер:", df.shape)
print(df.head())
print("\nПропуски:")
print(df.isna().sum())

## Очистка и подготовка

In [ ]:
df = df.dropna(subset=["lat", "lon"]).copy()
df = df.sort_values("timestamp").reset_index(drop=True)
df = df.drop_duplicates(subset="timestamp", keep="first").reset_index(drop=True)

print("После очистки:", df.shape)
print(df.head())

##Функции для расчёта (Haversine + Bearing)

In [ ]:
R_EARTH = 6371000.0  # радиус Земли в метрах

def haversine(lat1, lon1, lat2, lon2):
    """Расстояние между двумя точками (м) по формуле гаверсинуса."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

def bearing(lat1, lon1, lat2, lon2):
    """Направление движения в градусах: 0=север, 90=восток, 180=юг, 270=запад."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    theta = np.arctan2(x, y)
    return (np.degrees(theta) + 360) % 360

## Задачи 1–3. Расстояние, скорость, направление

In [ ]:
# Сдвинутые значения — координаты предыдущей точки
lat_prev = df["lat"].shift(1)
lon_prev = df["lon"].shift(1)
t_prev   = df["timestamp"].shift(1)

# 1) Расстояние от предыдущей точки (в метрах)
df["distant"] = haversine(lat_prev, lon_prev, df["lat"], df["lon"])

# 2) Скорость (м/с)
df["speed"] = df["distant"] / df["dt"]

# 4) Направление движения (в градусах)
df["direction"] = bearing(lat_prev, lon_prev, df["lat"], df["lon"])

print(df[["lat", "lon", "timestamp", "distant", "dt", "speed", "direction"]].head(10))

## Задача 4. Сокращённый анализ (≥ 4 параметра)

In [ ]:
stats = {
    "Количество точек":             len(df),
    "Общее время наблюдения, с":    int(df["timestamp"].max() - df["timestamp"].min()),
    "Общий пройденный путь, м":     round(df["distant"].sum(), 2),
    "Общий путь, км":               round(df["distant"].sum()/1000, 3),
    "Средняя скорость, м/с":        round(df["speed"].mean(), 3),
    "Средняя скорость, км/ч":       round(df["speed_kmh"].mean(), 2),
    "Медианная скорость, км/ч":     round(df["speed_kmh"].median(), 2),
    "Максимальная скорость, км/ч":  round(df["speed_kmh"].max(), 2),
    "Минимальная скорость, км/ч":   round(df["speed_kmh"].min(), 2),
    "Средний интервал, с":          round(df["dt"].mean(), 2),
    "Макс. интервал, с":            int(df["dt"].max()),
}

for k, v in stats.items():
    print(f"{k:35s}: {v}")